# ДЗ 11. Образовательный путь педагога

## 1. Кадровая витрина

Одна строка соответствует одному педагогу в одном году. Уход означает отсутствие в следующем срезе. Для 2024 года исход пока неизвестен.

In [1]:
from pathlib import Path

import pandas as pd

root = Path.cwd()
if root.name == 'HW':
    root = root.parent
data = root / 'data'
teachers = pd.read_parquet(data / 'teachers.parquet')
key = ['id', 'year']

assert not teachers.duplicated(key).any()
assert teachers.loc[teachers['year'] == 2024, 'target'].isna().all()

summary = teachers.groupby('year')['target'].agg(['size', 'count'])
summary.columns = ['Педагогов', 'Известно исходов']
summary['Уходов'] = teachers.groupby('year')['target'].sum(min_count=1).astype('Int64')
display(summary)

,Педагогов,Известно исходов,Уходов
year,,,
2022,17396,17396,2596
2023,17394,17394,2299
2024,17933,0,<NA>


## 2. Дипломы и курсы

Из двух книг читаем листы за три года. Сохраняем год листа отдельно от года окончания обучения.

In [2]:
def read_years(filename):
    sheets = pd.read_excel(data / filename, sheet_name=None, dtype={'id': str})
    parts = []
    for year in [2022, 2023, 2024]:
        part = sheets[str(year)]
        part['source_year'] = year
        parts.append(part)
    return pd.concat(parts, ignore_index=True)

edu = read_years('Образование.xlsx').rename(columns={
    'Тип образования': 'level', 'Наименование учебного заведения': 'university',
    'Дата окончания': 'end_year', 'Специализация': 'specialization',
    'Квалификация': 'qualification'
})
courses = read_years('ПК-ПП.xlsx').rename(columns={
    'Наименование программы': 'program',
    'Программа профессиональной переподготовки(ПП)/повышения квалификации(ПК)': 'kind',
    'Количество часов': 'hours', 'Дата получения документа/освоения программы': 'end_year'
})
display(edu['level'].value_counts(dropna=False), courses['kind'].value_counts(dropna=False))

level
Дополнительное профессиональное образование                   235012
Высшее образование                                             44894
Высшее образование - специалитет, магистратура                  8181
Среднее профессиональное образование                            8028
Высшее образование - бакалавриат                                5876
Среднее (полное) общее образование                               343
Аспирантура                                                      108
Среднее общее образование                                         98
Основное общее образование                                        66
Неполное высшее образование                                       66
Высшее образование - подготовка кадров высшей квалификации        47
Name: count, dtype: int64

kind
ПК    65670
ПП       63
Name: count, dtype: int64

Тип образования определяем по названию из выгрузки. Педагогический профиль ищем по тем же основам слов, что и в проекте. Это приближение, совпадение только по названию вуза слабее совпадения по квалификации.

In [3]:
for col in ['level', 'university', 'specialization', 'qualification']:
    edu[col] = edu[col].str.strip().replace({'не указано': pd.NA, '-': pd.NA, '': pd.NA})

edu['is_higher'] = edu['level'].str.startswith(('Высшее', 'Аспирантура'))
edu['is_dpo'] = edu['level'].str.startswith('Дополнительное')
fields = ['specialization', 'qualification', 'university']
words = 'педагог|учител|преподав'
edu['is_pedagogical'] = edu[fields].fillna('').agg(' '.join, axis=1).str.contains(words, case=False)
edu['is_pedagogical'] = edu['is_pedagogical'].astype('boolean').mask(edu[fields].isna().all(axis=1))

display(edu.groupby(['level', 'is_higher', 'is_dpo'], dropna=False).size().rename('Записей'))
examples = edu.groupby('is_pedagogical', dropna=False).head(2)
display(examples[['is_pedagogical', 'level'] + fields])

level                                                       is_higher  is_dpo
Аспирантура                                                 True       False        108
Высшее образование                                          True       False      44894
Высшее образование - бакалавриат                            True       False       5876
Высшее образование - подготовка кадров высшей квалификации  True       False         47
Высшее образование - специалитет, магистратура              True       False       8181
Дополнительное профессиональное образование                 False      True      235012
Неполное высшее образование                                 False      False         66
Основное общее образование                                  False      False         66
Среднее (полное) общее образование                          False      False        343
Среднее общее образование                                   False      False         98
Среднее профессиональное образование      

,is_pedagogical,level,specialization,qualification,university
0,True,"Высшее образование - специалитет, магистратура",Теория и методика преподавания иностранных язы...,"Лингвист, преподаватель","ФГОУ ВПО ""Государственный технологический унив..."
1,True,Высшее образование,Физика,учитель физики и математики,Бийский государственный педагогический институт
5,<NA>,Дополнительное профессиональное образование,NaN,NaN,NaN
10,False,Дополнительное профессиональное образование,Иностранный язык,NaN,NaN
12,False,Дополнительное профессиональное образование,NaN,Технологии дистанционного обучения с использов...,NaN
16,<NA>,Дополнительное профессиональное образование,NaN,NaN,NaN


Повтором считаем совпадение всех полей документа, кроме года листа. Оставляем первое появление. Разные программы одного года сохраняем.

In [4]:
def first_records(table):
    fields = table.columns.drop('source_year')
    return table.sort_values('source_year').drop_duplicates(subset=fields).copy()

courses['program'] = courses['program'].str.strip()
before = [len(edu), len(courses)]
edu = first_records(edu)
courses = first_records(courses)
counts = pd.DataFrame({'До': before, 'После': [len(edu), len(courses)]},
                      index=['Образование', 'ПК-ПП'])
counts['Повторов'] = counts['До'] - counts['После']
display(counts)

,До,После,Повторов
Образование,302719,104314,198405
ПК-ПП,65733,65721,12


## 3. Образовательный путь

Считаем дипломы ВО по книге об образовании и программы ПП по явной отметке в книге курсов. Записи ДПО из первой книги считаем отдельно: общего номера документа у двух источников нет.

Для давности и интервалов используем годы завершения из обеих книг. Пропуск, будущий год или окончание до 14 лет исключаем из расчёта дат. Возраст проверяем, если он известен на год листа.

In [5]:
ages = teachers[['id', 'year', 'age']].rename(columns={'year': 'source_year'})

def check_dates(table):
    table = table.merge(ages, on=['id', 'source_year'], how='left', validate='many_to_one')
    end = pd.to_numeric(table['end_year'], errors='coerce')
    first_possible = table['source_year'] - table['age'] + 14
    wrong = (end > table['source_year']) | (end < first_possible) | (end < 1900)
    table['event_year'] = end.mask(wrong)
    return table

edu = check_dates(edu)
courses = check_dates(courses)
display(pd.Series({'Образование': edu['event_year'].isna().sum(),
                   'ПК-ПП': courses['event_year'].isna().sum()}, name='Без надёжной даты'))

Образование    396
ПК-ПП            0
Name: Без надёжной даты, dtype: int64

Для каждого среза берём только уже появившиеся записи. Интервал считаем между разными годами обучения. Если известен один год, давность посчитать можно, а интервал пока неизвестен.

In [6]:
history = []
for year in [2022, 2023, 2024]:
    diplomas = edu[edu['source_year'] <= year]
    programs = courses[courses['source_year'] <= year]
    part = teachers.loc[teachers['year'] == year, key].set_index('id').copy()

    part['n_higher'] = diplomas[diplomas['is_higher']].groupby('id').size()
    pedagogical = diplomas['is_higher'] & diplomas['is_pedagogical']
    part['n_higher_pedagogical'] = diplomas[pedagogical].groupby('id').size()
    part['n_dpo_records'] = diplomas[diplomas['is_dpo']].groupby('id').size()
    part['n_pp'] = programs[programs['kind'] == 'ПП'].groupby('id').size()
    count_cols = ['n_higher', 'n_higher_pedagogical', 'n_dpo_records', 'n_pp']
    part[count_cols] = part[count_cols].fillna(0).astype(int)

    dates = pd.concat([diplomas[['id', 'event_year']], programs[['id', 'event_year']]])
    dates = dates.dropna().drop_duplicates().sort_values(['id', 'event_year'])
    years = dates.groupby('id')['event_year']
    part['years_since_last'] = year - years.max()
    part['learning_span'] = years.max() - years.min()
    dates['gap'] = years.diff()
    part['max_learning_gap'] = dates.groupby('id')['gap'].max()
    history.append(part.reset_index())

education_by_year = pd.concat(history, ignore_index=True)
teachers_hw11 = teachers.merge(education_by_year, on=key, how='left', validate='one_to_one')
assert teachers_hw11[key + ['target']].equals(teachers[key + ['target']])
display(education_by_year.groupby('year').mean(numeric_only=True).round(4))
display(education_by_year.groupby('year')[['years_since_last', 'max_learning_gap']].count())

,n_higher,n_higher_pedagogical,n_dpo_records,n_pp,years_since_last,learning_span,max_learning_gap
year,,,,,,,
2022,1.0748,0.8524,2.8714,0.0009,1.1688,21.4840,15.6489
2023,1.1164,0.8781,3.2557,0.0025,0.8705,21.9524,15.1906
2024,1.1682,0.9142,3.4966,0.0029,0.8870,22.0281,14.8538


,years_since_last,max_learning_gap
year,,
2022,17329,16533
2023,17311,16840
2024,17860,17319


### Проверка на одном педагоге

Выберем педагога с несколькими годами обучения. Сверим последнюю дату и максимальный перерыв с его списком лет за срез 2023 года.

In [7]:
sample = education_by_year.query('year == 2023 and n_higher > 0 and max_learning_gap > 0').iloc[0]
sample_id = sample['id']
known_years = []
for table in [edu, courses]:
    rows = table[(table['id'] == sample_id) & (table['source_year'] <= 2023)]
    known_years.extend(rows['event_year'].dropna().astype(int))
known_years = sorted(set(known_years))
gaps = [later - earlier for earlier, later in zip(known_years, known_years[1:])]
print('Годы обучения:', known_years)
print('Давность:', 2023 - known_years[-1], 'Максимальный перерыв:', max(gaps))
assert sample['years_since_last'] == 2023 - known_years[-1]
assert sample['max_learning_gap'] == max(gaps)
display(education_by_year[education_by_year['id'] == sample_id].drop(columns='id'))

Годы обучения: [1993, 1999, 2009, 2010, 2012, 2019, 2020, 2021, 2022, 2023]
Давность: 0 Максимальный перерыв: 10


,year,n_higher,n_higher_pedagogical,n_dpo_records,n_pp,years_since_last,learning_span,max_learning_gap
7611,2022,1,1,8,0,0.0,29.0,10.0
17396,2023,1,1,8,0,0.0,30.0,10.0
34790,2024,1,1,8,0,0.0,31.0,10.0


В примере за 2023 год последнее обучение прошло в самом году среза. Самый большой перерыв составил 10 лет, с 1999 по 2009 год. Для дальнейших шагов таблица с признаками доступна в `teachers_hw11`.

## 4. Вузы в рейтинге RAEX-100 за 2022 год

Берём место в RAEX-100 за 2022 год и название вуза. Меньший номер означает более высокое место. Для всех трёх срезов используем один выпуск, уже доступный к концу 2022 года.

In [8]:
import re

raex = pd.read_csv(root / 'HW' / 'data' / 'raex_2022.csv')
assert raex['rank'].tolist() == list(range(1, 101))
display(raex.head())

,rank,university
0,1,Московский государственный университет имени М...
1,2,Московский физико-технический институт (национ...
2,3,Санкт-Петербургский государственный университет
3,4,Национальный исследовательский ядерный универс...
4,5,"Национальный исследовательский университет ""Вы..."


Сравниваем полные названия после удаления кавычек, лишних пробелов и обозначений вроде «ФГБОУ ВО». Частые сокращения задаём явно. Например, МПГУ занимает место 35, а МГПУ занимает место 64. Созвучные названия и филиалы без точного соответствия остаются без позиции.

In [9]:
def university_key(name):
    text = name.lower().replace('ё', 'е')
    text = re.sub(r'[^а-яa-z0-9]+', ' ', text).strip()
    text = re.sub(r'^(фгбоу|фгаоу|гбоу|гаоу|гоу|фгоу) (во |впо )?', '', text)
    text = re.sub(r'^(города москвы|г москвы) ', '', text)
    text = re.sub(r' г москва$', '', text)
    return re.sub(r'\bим\b', 'имени', text).strip()

rank_by_name = dict(zip(raex['university'].map(university_key), raex['rank']))
assert len(rank_by_name) == 100

aliases = {
    'МПГУ': 'Московский педагогический государственный университет',
    'МГПУ': 'Московский городской педагогический университет',
    'МГЛУ': 'Московский государственный лингвистический университет',
    'МГППУ': 'Московский государственный психолого-педагогический университет',
    'МГУ им. М.В. Ломоносова': 'Московский государственный университет имени М.В. Ломоносова',
    'РГГУ': 'Российский государственный гуманитарный университет',
    'РГПУ им. А.И. Герцена': 'Российский государственный педагогический университет имени А. И. Герцена',
    'РУДН': 'Российский университет дружбы народов имени Патриса Лумумбы',
    'Российский университет дружбы народов': 'Российский университет дружбы народов имени Патриса Лумумбы',
    'НИЯУ МИФИ': 'Национальный исследовательский ядерный университет «МИФИ»',
    'НИУ ВШЭ': 'Национальный исследовательский университет "Высшая школа экономики"',
    'МГТУ им. Н.Э. Баумана': 'Московский государственный технический университет имени Н.Э. Баумана (национальный исследовательский университет)',
    'МФТИ': 'Московский физико-технический институт (национальный исследовательский университет)',
    'Московский физико-технический институт': 'Московский физико-технический институт (национальный исследовательский университет)',
    'Московский авиационный институт': 'Московский авиационный институт (национальный исследовательский университет)',
}
for alias, official_name in aliases.items():
    rank = rank_by_name[university_key(official_name)]
    alias_key = university_key(alias)
    assert rank_by_name.get(alias_key, rank) == rank
    rank_by_name[alias_key] = rank

degrees = edu.loc[edu['is_higher'], ['id', 'source_year', 'university']].copy()
degrees['name_key'] = degrees['university'].map(university_key, na_action='ignore')
degrees['raex_rank_2022'] = degrees['name_key'].map(rank_by_name).astype('Int64')

matches = degrees.dropna(subset=['raex_rank_2022']).groupby(
    ['university', 'raex_rank_2022']).size().nlargest(10)
display(matches.rename('Дипломов').to_frame())
display(degrees.loc[degrees['raex_rank_2022'].isna(), 'university']
        .value_counts().head(10).rename('Дипломов без совпадения').to_frame())

,,Дипломов
university,raex_rank_2022,
Московский педагогический государственный университет,35,919
Московский городской педагогический университет,64,707
"ФГБОУ ВО ""Московский педагогический государственный университет""",35,623
МПГУ,35,358
"ГАОУ ВО ""Московский городской педагогический университет""",64,267
МГПУ,64,250
"ГОУ ВПО ""Московский педагогический государственный университет""",35,205
"ФГБОУ ВПО ""Московский педагогический государственный университет""",35,188
Московский государственный лингвистический университет,34,113


,Дипломов без совпадения
university,
Московский государственный заочный педагогический институт,374
Московский государственный открытый педагогический университет,266
Московский государственный областной университет,195
Московский городской психолого-педагогический университет,138
Московский государственный открытый педагогический институт,117
Московский педагогический университет,113
Московский гуманитарный педагогический институт,111
ГОУ ВПО Московский государственный областной университет,87
МГПИ им. В.И. Ленина,80


Для каждого педагога выбираем лучшую позицию среди дипломов ВО, уже известных на год среза. Если позиция неизвестна, сохраняем пропуск. Покрытие считаем отдельно для дипломов и педагогов.

In [10]:
rank_parts = []
coverage = []
for year in [2022, 2023, 2024]:
    cohort = teachers.loc[teachers['year'] == year, ['id', 'year']].copy()
    known = degrees[(degrees['source_year'] <= year) & degrees['id'].isin(cohort['id'])]
    best = known.groupby('id')['raex_rank_2022'].min()
    cohort['raex_rank_2022'] = cohort['id'].map(best)
    cohort['raex_status_2022'] = 'нет диплома ВО'
    cohort.loc[cohort['id'].isin(known['id']), 'raex_status_2022'] = 'нет названия'
    named = known.dropna(subset=['university'])
    cohort.loc[cohort['id'].isin(named['id']), 'raex_status_2022'] = 'нет совпадения'
    cohort.loc[cohort['raex_rank_2022'].notna(), 'raex_status_2022'] = 'есть позиция'
    rank_parts.append(cohort)

    coverage.append({
        'Год': year, 'Дипломов ВО': len(known),
        'С позицией, дипломов': known['raex_rank_2022'].count(),
        '% всех дипломов ВО': round(100 * known['raex_rank_2022'].notna().mean(), 1),
        'Педагогов': len(cohort), 'С позицией, педагогов': cohort['raex_rank_2022'].count(),
        '% всех педагогов': round(100 * cohort['raex_rank_2022'].notna().mean(), 1)
    })

rank_history = pd.concat(rank_parts, ignore_index=True)
teachers_hw11 = teachers_hw11.drop(columns=['raex_rank_2022', 'raex_status_2022'], errors='ignore')
teachers_hw11 = teachers_hw11.merge(rank_history, on=['id', 'year'], how='left', validate='one_to_one')
assert teachers_hw11[['id', 'year', 'target']].equals(teachers[['id', 'year', 'target']])
assert teachers_hw11['raex_status_2022'].notna().all()
display(pd.DataFrame(coverage).set_index('Год'))
display(pd.crosstab(teachers_hw11['year'], teachers_hw11['raex_status_2022']))

,Дипломов ВО,"С позицией, дипломов",% всех дипломов ВО,Педагогов,"С позицией, педагогов",% всех педагогов
Год,,,,,,
2022,18698,4804,25.7,17396,4233,24.3
2023,19418,5089,26.2,17394,4343,25.0
2024,20950,5739,27.4,17933,4714,26.3


raex_status_2022,есть позиция,нет диплома ВО,нет названия,нет совпадения
year,,,,
2022,4233,883,3,12277
2023,4343,823,3,12225
2024,4714,781,1,12437


## 5. Проверка признаков

Сравним накопленные счётчики с флагами текущего среза. Например, два диплома дают значение 2 в счётчике и 1 во флаге. Для переподготовки счётчик учитывает только явные ПП из книги курсов, а флаг проекта также учитывает сведения из книги об образовании.

In [11]:
import json

with open(root / 'models' / 'metadata.json', encoding='utf-8') as file:
    metadata = json.load(file)

base = metadata['features']
history_features = ['n_higher', 'n_higher_pedagogical', 'n_dpo_records', 'n_pp',
                    'years_since_last', 'learning_span', 'max_learning_gap']
rating_features = ['raex_rank_2022', 'raex_status_2022']
variants = {'A': base, 'B': base + history_features, 'C': base + history_features + rating_features}

checks = []
for year, group in teachers_hw11.groupby('year'):
    for count, flag in [('n_higher', 'edu_higher'), ('n_higher_pedagogical', 'edu_pedagogical'),
                        ('n_pp', 'retraining')]:
        checks.append({'Год': year, 'Счётчик': count, 'Флаг': flag,
                       'Счётчик больше 1': (group[count] > 1).sum(),
                       'Наличие расходится с флагом': (group[count].gt(0) != group[flag]).sum()})
display(pd.DataFrame(checks))
display(teachers_hw11.groupby('year')[history_features + rating_features].count().T)
print('A:', ', '.join(base))
print('B добавляет:', ', '.join(history_features))
print('C добавляет:', ', '.join(rating_features))

,Год,Счётчик,Флаг,Счётчик больше 1,Наличие расходится с флагом
0,2022,n_higher,edu_higher,2041,0
1,2022,n_higher_pedagogical,edu_pedagogical,929,0
2,2022,n_pp,retraining,0,6631
3,2023,n_higher,edu_higher,2502,1
4,2023,n_higher_pedagogical,edu_pedagogical,1284,6
5,2023,n_pp,retraining,0,6850
6,2024,n_higher,edu_higher,3154,9
7,2024,n_higher_pedagogical,edu_pedagogical,1796,22
8,2024,n_pp,retraining,0,7085


year,2022,2023,2024
n_higher,17396,17394,17933
n_higher_pedagogical,17396,17394,17933
n_dpo_records,17396,17394,17933
n_pp,17396,17394,17933
years_since_last,17329,17311,17860
learning_span,17329,17311,17860
max_learning_gap,16533,16840,17319
raex_rank_2022,4233,4343,4714
raex_status_2022,17396,17394,17933


A: age, experience_age_pedagogic_clean, experience_age_other_clean, n_subjects, n_parallels, min_parallel, max_parallel, sex, class_manager, edu_higher, edu_pedagogical, pk_recent, pk_this_year, retraining, main_position_cat, academic_degree_cat, category_qual_cat, district
B добавляет: n_higher, n_higher_pedagogical, n_dpo_records, n_pp, years_since_last, learning_span, max_learning_gap
C добавляет: raex_rank_2022, raex_status_2022


Давность и интервалы измеряются в годах. Проектные флаги ПК показывают наличие события в текущем году или в недавнем окне. Ноль в давности возможен также после окончания вуза. Числовые пропуски передаём CatBoost напрямую. В выборке остаются все педагоги, включая тех, чей вуз не удалось сопоставить.

## 6. Сравнение моделей

Берём параметры CatBoost из метаданных проекта. Меняем только признаки. Для каждого варианта по трём фолдам 2022 года выбираем порог с максимальной precision при recall от 0,70. Затем обучаем модель на всём 2022 году и оцениваем на 2023 году. Основная метрика AP показывает качество ранжирования редкого класса. Для Brier меньшее значение лучше.

### Нагрузка и срок категории

Проверим ещё два признака поверх 18 признаков варианта A. Первый показывает число разных сочетаний предмета и параллели у педагога. Второй показывает, сколько лет от года среза до указанного окончания категории. Ноль означает окончание в год среза, отрицательное число означает, что указанный срок уже прошёл. Если год окончания неизвестен, сохраняем пропуск.

Вариант D добавляет нагрузку, E добавляет срок категории, F добавляет оба признака.

In [12]:
teachers_hw11['years_to_category_end'] = teachers_hw11['category_due_clean'] - teachers_hw11['year']
variants['D'] = base + ['n_subject_parallel']
variants['E'] = base + ['years_to_category_end']
variants['F'] = base + ['n_subject_parallel', 'years_to_category_end']

display(teachers_hw11.groupby('year').agg(
    педагогов=('id', 'size'),
    известна_нагрузка=('n_subject_parallel', 'count'),
    известен_срок=('years_to_category_end', 'count')
))

,педагогов,известна_нагрузка,известен_срок
year,,,
2022,17396,17396,13608
2023,17394,17394,13218
2024,17933,17933,13327


In [13]:
import numpy as np
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (average_precision_score, roc_auc_score, brier_score_loss,
                             precision_recall_curve, precision_score, recall_score)

train = teachers_hw11.query('year == 2022').reset_index(drop=True)
test = teachers_hw11.query('year == 2023').reset_index(drop=True)
y_train = train['target'].astype(int)
y_test = test['target'].astype(int)
folds = list(StratifiedKFold(3, shuffle=True, random_state=42).split(train, y_train))
print('Обучение:', len(train), 'Проверка:', len(test))
print('Доля уходов на проверке:', round(y_test.mean(), 4))

Обучение: 17396 Проверка: 17394
Доля уходов на проверке: 0.1322


In [14]:
results = []
for variant, features in variants.items():
    params = metadata['model_params'].copy()
    if variant == 'C':
        params['cat_features'] = metadata['categorical_features'] + ['raex_status_2022']
    X = train[features].copy()
    X_test = test[features].copy()
    if 'raex_rank_2022' in features:
        X['raex_rank_2022'] = X['raex_rank_2022'].astype(float)
        X_test['raex_rank_2022'] = X_test['raex_rank_2022'].astype(float)

    oof = np.zeros(len(train))
    for fit, valid in folds:
        model = CatBoostClassifier(**params)
        model.fit(X.iloc[fit], y_train.iloc[fit])
        oof[valid] = model.predict_proba(X.iloc[valid])[:, 1]

    precision, recall, thresholds = precision_recall_curve(y_train, oof)
    allowed = np.flatnonzero(recall[:-1] >= 0.70)
    threshold = thresholds[allowed[np.argmax(precision[allowed])]]

    model = CatBoostClassifier(**params)
    model.fit(X, y_train)
    probability = model.predict_proba(X_test)[:, 1]
    prediction = probability >= threshold
    results.append({'Вариант': variant, 'Признаков': len(features),
                    'AP': average_precision_score(y_test, probability),
                    'ROC-AUC': roc_auc_score(y_test, probability),
                    'Brier': brier_score_loss(y_test, probability),
                    'Порог': threshold, 'Precision': precision_score(y_test, prediction),
                    'Recall': recall_score(y_test, prediction)})

scores = pd.DataFrame(results).set_index('Вариант')
display(scores.round(5))
reference = metadata['metrics']
assert np.allclose(scores.loc['A', ['AP', 'ROC-AUC', 'Brier']],
                   [reference['ap'], reference['roc_auc'], reference['brier']], atol=1e-5)

,Признаков,AP,ROC-AUC,Brier,Порог,Precision,Recall
Вариант,,,,,,,
A,18,0.29673,0.72329,0.10539,0.13197,0.21788,0.71988
B,25,0.29429,0.72214,0.10549,0.13228,0.22033,0.71553
C,27,0.30104,0.72354,0.10506,0.13156,0.21857,0.70944
D,19,0.29872,0.72431,0.10521,0.13280,0.21856,0.71814
E,19,0.29855,0.72504,0.10520,0.13214,0.21770,0.71814
F,20,0.29960,0.72571,0.10514,0.13374,0.21855,0.71857


## 7. Что дало обогащение

Сравним разности на одном наборе педагогов. B минус A показывает вклад истории. C минус B показывает добавочный вклад рейтинга и статуса сопоставления.

In [15]:
metrics = scores[['AP', 'ROC-AUC', 'Brier']]
changes = pd.DataFrame({'B минус A': metrics.loc['B'] - metrics.loc['A'],
                        'C минус B': metrics.loc['C'] - metrics.loc['B'],
                        'C минус A': metrics.loc['C'] - metrics.loc['A']}).T
display(changes.round(5))

,AP,ROC-AUC,Brier
B минус A,-0.00244,-0.00115,0.00010
C минус B,0.00675,0.00141,-0.00043
C минус A,0.00431,0.00025,-0.00033


**Вывод.** В 2023 году у 2 502 педагогов было больше одного диплома ВО. Счётчики дают дополнительную информацию, но семь признаков образовательного пути сами по себе не улучшили прогноз. AP снизилась с 0,29673 до 0,29429, Brier вырос с 0,10539 до 0,10549.

После добавления позиции RAEX и статуса сопоставления AP выросла до 0,30104. Прирост составил 0,00675 относительно B и 0,00431 относительно A. Среди A, B и C у варианта C наименьший Brier (0,10506). При порогах, выбранных на 2022 году, recall у C составил 0,7094 против 0,7199 у A.

По основной метрике среди A, B и C лучший результат у C. Позиция вуза известна у 4 343 из 17 394 педагогов проверочного года (25,0%). Мы проверяли позицию и статус вместе, поэтому их отдельный вклад неизвестен. Небольшой прирост наблюдался на одном ранее изученном годе.

### Что добавили нагрузка и срок категории

In [16]:
extra = scores.loc[['A', 'D', 'E', 'F']].copy()
extra['Разница AP с A'] = extra['AP'] - extra.loc['A', 'AP']
extra.index = ['База', 'База + нагрузка', 'База + срок категории', 'База + оба признака']
display(extra.round(5))

,Признаков,AP,ROC-AUC,Brier,Порог,Precision,Recall,Разница AP с A
База,18,0.29673,0.72329,0.10539,0.13197,0.21788,0.71988,0.00000
База + нагрузка,19,0.29872,0.72431,0.10521,0.13280,0.21856,0.71814,0.00199
База + срок категории,19,0.29855,0.72504,0.10520,0.13214,0.21770,0.71814,0.00181
База + оба признака,20,0.29960,0.72571,0.10514,0.13374,0.21855,0.71857,0.00287


Оба признака по отдельности улучшили AP относительно базы. Вместе они дали больший прирост: AP выросла с 0,29673 до 0,29960 (+0,00287), ROC-AUC с 0,72329 до 0,72571, а Brier снизился с 0,10539 до 0,10514. При пороге, выбранном на 2022 году, полнота составила 71,86% против 71,99% у базы.

Среди всех шести вариантов у C с образовательной историей и RAEX выше всего AP (0,30104), а у F с нагрузкой и сроком категории выше всего ROC-AUC (0,72571). Два доступных в витрине признака стоит оставить кандидатами для дальнейшей проверки. Пока сравнение охватывает один проверочный год.